In [45]:
import pandas as pd
import os

# ============================
# Paths
# ============================
super_path = "/Users/judycheng/Desktop/supercharger in washington state.xls"
output_path = "/Users/judycheng/Desktop/supercharger_by_county_summary.xlsx"

# ============================
# Read file
# ============================
df = pd.read_excel(super_path)

# ============================
# Filter Washington
# ============================
df_wa = df[df["State"] == "Washington"]

# ============================
# Count all charge *points*  
# (1 row = 1 charger point)
# ============================
counts = (
    df_wa.groupby("County")["County"]
    .count()
    .reset_index(name="Supercharger_Count")
)

# ============================
# Clean "County" suffix
# ============================
counts["County"] = counts["County"].str.replace(" County", "", regex=False)

# Sort descending
counts = counts.sort_values("Supercharger_Count", ascending=False)

# ============================
# Export to Excel
# ============================
counts.to_excel(output_path, index=False)

print("Done! File exported to:", output_path)


Done! File exported to: /Users/judycheng/Desktop/supercharger_by_county_summary.xlsx


In [42]:
import pandas as pd

# =========================
# Paths
# =========================
residents_path = "/Users/judycheng/Desktop/Population 2024 age 25 to 59.xlsx"
super_summary_path = "/Users/judycheng/Desktop/supercharger_by_county_summary.xlsx"
ev_path = "/Users/judycheng/Desktop/coordinates_output.xlsm"

output_forecast_path = (
    "/Users/judycheng/Desktop/wa_county_ev_forecast_baseline_2024_to_2050.xlsx"
)

# =========================
# 1) Read Residents (Population) file
# =========================
residents_df = pd.read_excel(residents_path)

# Clean County names
residents_df["County"] = (
    residents_df["County"].astype(str)
    .str.replace(" County", "", regex=False)
    .str.strip()
)

# ❗ Explicitly use the "total" column as Pop_2024
if "total" not in residents_df.columns:
    raise ValueError("The residents file does not contain a 'total' column.")

pop_df = residents_df[["County", "total"]].rename(columns={"total": "Pop_2024"})

# =========================
# 2) Read Supercharger summary (points per county)
# =========================
super_df = pd.read_excel(super_summary_path)
super_df["County"] = super_df["County"].astype(str).str.strip()

super_df = super_df.rename(columns={"Supercharger_Count": "Superchargers_2024"})
super_df = super_df[["County", "Superchargers_2024"]]

# =========================
# 3) Read EV registration file (VIN count)
# =========================
ev_df = pd.read_excel(ev_path)
ev_df["County"] = (
    ev_df["County"].astype(str)
    .str.replace(" County", "", regex=False)
    .str.strip()
)

ev_counts = (
    ev_df.groupby("County")["County"]
    .count()
    .reset_index(name="EVs_2024")
)

# =========================
# 4) Build 2024 baseline
# =========================
base = pop_df.merge(super_df, on="County", how="left")
base = base.merge(ev_counts, on="County", how="left")

base["Superchargers_2024"] = base["Superchargers_2024"].fillna(0).astype(int)
base["EVs_2024"] = base["EVs_2024"].fillna(0).astype(int)

base["Adoption_2024"] = (
    base["EVs_2024"] / base["Pop_2024"]
).fillna(0)

# Order columns
base = base[["County", "Pop_2024", "Superchargers_2024", "EVs_2024", "Adoption_2024"]]

# =========================
# 5) Add 2025–2050 empty forecast columns
# =========================
for year in range(2025, 2051):
    base[f"Superchargers_{year}"] = pd.NA
    base[f"EVs_{year}"] = pd.NA
    base[f"Adoption_{year}"] = pd.NA

# Sort alphabetically
base = base.sort_values("County").reset_index(drop=True)

# =========================
# 6) Export to Excel
# =========================
base.to_excel(output_forecast_path, index=False)

print("✅ Forecast baseline file created:")
print(output_forecast_path)


✅ Forecast baseline file created:
/Users/judycheng/Desktop/wa_county_ev_forecast_baseline_2024_to_2050.xlsx


In [46]:
import pandas as pd

# ========================================================
# FILE PATHS
# ========================================================
base_path = "/Users/judycheng/Desktop/wa_county_ev_forecast_baseline_2024_to_2050.xlsx"

king_path   = "/Users/judycheng/Desktop/king_county_ev_projection_mc_monotonic.xlsx"
pierce_path = "/Users/judycheng/Desktop/pierce_county_ev_projection_mc_monotonic.xlsx"
kitsap_path = "/Users/judycheng/Desktop/kitsap_county_ev_projection_mc_monotonic.xlsx"
chelan_path = "/Users/judycheng/Desktop/chelan_county_ev_projection_mc_monotonic.xlsx"

output_path = "/Users/judycheng/Desktop/wa_county_ev_forecast_filled_mc.xlsx"

YEARS = list(range(2025, 2051))


# ========================================================
# 1. LOAD BASELINE FILE
# ========================================================
df = pd.read_excel(base_path)
df["County"] = df["County"].astype(str).str.strip()


# ========================================================
# 2. LOAD MC TEMPLATE FUNCTION
# ========================================================
def load_mc(path):
    mc = pd.read_excel(path, sheet_name="Forecast")
    mc = mc.rename(columns={mc.columns[0]: "Year"})
    mc = mc.set_index("Year")

    needed = ["Forecast_Chargers", "Forecast_EVs_P50", "Forecast_Adoption_P50"]
    for c in needed:
        if c not in mc.columns:
            raise ValueError(f"Missing column {c} in template: {path}")

    return mc[needed]


# Load all 4 MC templates
king_mc   = load_mc(king_path)
pierce_mc = load_mc(pierce_path)
kitsap_mc = load_mc(kitsap_path)
chelan_mc = load_mc(chelan_path)


# ========================================================
# 3. SELECT TEMPLATE BASED ON POPULATION
# ========================================================
def select_template(pop):
    if pop > 1_000_000:
        return king_mc
    elif pop > 130_000:
        return pierce_mc
    elif pop > 34_000:
        return kitsap_mc
    else:
        return chelan_mc


# ========================================================
# 4. FILL IN FORECAST VALUES FOR 2025–2050
# ========================================================
for idx, row in df.iterrows():
    pop = row["Pop_2024"]
    template = select_template(pop)

    for year in YEARS:

        # Read adoption and chargers from MC template
        adoption = template.loc[year, "Forecast_Adoption_P50"]
        chargers = template.loc[year, "Forecast_Chargers"]

        # EVs = Adoption × Pop_2024
        evs = adoption * pop

        df.at[idx, f"Adoption_{year}"]      = adoption
        df.at[idx, f"Superchargers_{year}"] = chargers
        df.at[idx, f"EVs_{year}"]           = evs


# Optional: convert numeric
for year in YEARS:
    df[f"Superchargers_{year}"] = pd.to_numeric(df[f"Superchargers_{year}"], errors="coerce")
    df[f"EVs_{year}"]           = pd.to_numeric(df[f"EVs_{year}"], errors="coerce")
    df[f"Adoption_{year}"]      = pd.to_numeric(df[f"Adoption_{year}"], errors="coerce")


# ========================================================
# 5. EXPORT FINAL FILE
# ========================================================
df.to_excel(output_path, index=False)

print("✅ Monte Carlo forecast (2025–2050) successfully filled.")
print("📄 Output saved to:")
print(output_path)


✅ Monte Carlo forecast (2025–2050) successfully filled.
📄 Output saved to:
/Users/judycheng/Desktop/wa_county_ev_forecast_filled_mc.xlsx


In [47]:
import pandas as pd

# ========================================================
# FILE PATHS
# ========================================================
base_path = "/Users/judycheng/Desktop/wa_county_ev_forecast_baseline_2024_to_2050.xlsx"

king_path   = "/Users/judycheng/Desktop/king_county_ev_projection_mc_monotonic.xlsx"
pierce_path = "/Users/judycheng/Desktop/pierce_county_ev_projection_mc_monotonic.xlsx"
kitsap_path = "/Users/judycheng/Desktop/kitsap_county_ev_projection_mc_monotonic.xlsx"
chelan_path = "/Users/judycheng/Desktop/chelan_county_ev_projection_mc_monotonic.xlsx"

output_path = "/Users/judycheng/Desktop/wa_county_ev_forecast_filled_mc.xlsx"

YEARS = list(range(2025, 2051))


# ========================================================
# 1. LOAD BASELINE FILE
# ========================================================
df = pd.read_excel(base_path)
df["County"] = df["County"].astype(str).str.strip()


# ========================================================
# 2. MC TEMPLATE LOADER
# ========================================================
def load_mc(path):
    mc = pd.read_excel(path, sheet_name="Forecast")
    mc = mc.rename(columns={mc.columns[0]: "Year"})
    mc = mc.set_index("Year")

    needed = ["Forecast_Chargers", "Forecast_EVs_P50", "Forecast_Adoption_P50"]
    for c in needed:
        if c not in mc.columns:
            raise ValueError(f"Missing column {c} in template: {path}")

    return mc[needed]


# Load templates
king_mc   = load_mc(king_path)
pierce_mc = load_mc(pierce_path)
kitsap_mc = load_mc(kitsap_path)
chelan_mc = load_mc(chelan_path)


# ========================================================
# 3. SELECT TEMPLATE BASED ON POPULATION
# ========================================================
def select_template(pop):
    if pop > 1_000_000:
        return king_mc
    elif pop > 130_000:
        return pierce_mc
    elif pop > 34_000:
        return kitsap_mc
    else:
        return chelan_mc


# ========================================================
# 4. FILL FORECAST (2025–2050)
# ========================================================
for idx, row in df.iterrows():
    pop = row["Pop_2024"]
    template = select_template(pop)

    for year in YEARS:
        adoption = template.loc[year, "Forecast_Adoption_P50"]
        chargers = template.loc[year, "Forecast_Chargers"]
        evs = adoption * pop

        df.at[idx, f"Adoption_{year}"]      = adoption
        df.at[idx, f"Superchargers_{year}"] = chargers
        df.at[idx, f"EVs_{year}"]           = evs


# Convert to numeric
for year in YEARS:
    df[f"Superchargers_{year}"] = pd.to_numeric(df[f"Superchargers_{year}"], errors="coerce")
    df[f"EVs_{year}"]           = pd.to_numeric(df[f"EVs_{year}"], errors="coerce")
    df[f"Adoption_{year}"]      = pd.to_numeric(df[f"Adoption_{year}"], errors="coerce")


# ========================================================
# 5. ADD TOTAL ROW AT BOTTOM
# ========================================================

total = {}

# County label
total["County"] = "TOTAL"

# Pop_2024 sum
total["Pop_2024"] = df["Pop_2024"].sum()

# Sum EV and Charger columns & weighted average for Adoption
for col in df.columns:
    if col.startswith("EVs_") or col.startswith("Superchargers_"):
        total[col] = df[col].sum(skipna=True)

    elif col.startswith("Adoption_"):
        year = col.split("_")[1]

        # Weighted average:
        # sum(pop * adoption) / sum(pop)
        weighted = (df["Pop_2024"] * df[col]).sum() / df["Pop_2024"].sum()
        total[col] = weighted


# Append row
df = pd.concat([df, pd.DataFrame([total])], ignore_index=True)


# ========================================================
# 6. EXPORT FINAL FILE
# ========================================================
df.to_excel(output_path, index=False)

print("✅ Monte Carlo forecast (2025–2050) successfully filled.")
print("➕ TOTAL row added.")
print("📄 Output saved to:")
print(output_path)


✅ Monte Carlo forecast (2025–2050) successfully filled.
➕ TOTAL row added.
📄 Output saved to:
/Users/judycheng/Desktop/wa_county_ev_forecast_filled_mc.xlsx


In [7]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
from openpyxl import load_workbook
from openpyxl.drawing.image import Image as XLImage
from matplotlib.ticker import PercentFormatter

# ======================================================
# FILE PATHS
# ======================================================
excel_path = "/Users/judycheng/Desktop/wa_county_ev_forecast_filled_mc.xlsx"
desktop = os.path.join(os.path.expanduser("~"), "Desktop")

chart1_png = os.path.join(desktop, "chart1_adoption_evs_vs_year.png")
chart2_png = os.path.join(desktop, "chart2_chargers_vs_year.png")
chart3_png = os.path.join(desktop, "chart3_adopt_evs_vs_chargers.png")

# ======================================================
# 1. LOAD FINAL FILE & TOTAL ROW
# ======================================================
df = pd.read_excel(excel_path)
total = df[df["County"] == "TOTAL"].iloc[0]

years = list(range(2024, 2051))

# Series from TOTAL row
adopt = [total[f"Adoption_{y}"] for y in years]
evs = [total[f"EVs_{y}"] for y in years]
# 🚩 FIX: use TOTAL superchargers directly (already the stock level by year)
chargers = [total.get(f"Superchargers_{y}", 0) for y in years]

# For sanity check (optional)
print(f"Max chargers (should be ~1259): {max(chargers)}")

# ======================================================
# 2. CREATE CHARTS (dots only) + SAVE AS PNG
# ======================================================

# -------- Chart 1 --------
# Adoption rate & EVs vs Year
plt.figure(figsize=(10, 6))
plt.scatter(years, evs, s=60, label="EV Registrations")

ax1 = plt.gca()
ax2 = ax1.twinx()
ax2.scatter(years, adopt, s=60, marker="x", label="Adoption Rate")

ax1.set_title("Statewide Adoption Rate & EV Registrations vs Year (2024–2050)")
ax1.set_xlabel("Year")
ax1.set_ylabel("EV Registrations")
ax2.set_ylabel("Adoption Rate")
ax2.yaxis.set_major_formatter(PercentFormatter(1.0))

ax1.grid(True)

# Combined legend
h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc="best")

plt.savefig(chart1_png, dpi=300, bbox_inches="tight")
plt.close()


# -------- Chart 2 --------
# Total Superchargers vs Year (NO extra cumsum)
plt.figure(figsize=(10, 6))
plt.scatter(years, chargers, s=60)

plt.title("Total Superchargers vs Year (2024–2050)")
plt.xlabel("Year")
plt.ylabel("Total Superchargers")
plt.grid(True)

plt.savefig(chart2_png, dpi=300, bbox_inches="tight")
plt.close()


# -------- Chart 3 --------
# EV Registrations (left Y) & Adoption Rate (right Y) vs Total Superchargers (X)
plt.figure(figsize=(10, 6))

x = chargers

ax1 = plt.gca()
ax1.scatter(x, evs, s=60, label="EV Registrations")
ax1.set_xlabel("Total Superchargers")
ax1.set_ylabel("EV Registrations")
ax1.grid(True)

ax2 = ax1.twinx()
ax2.scatter(x, adopt, s=60, marker="x", label="Adoption Rate")
ax2.set_ylabel("Adoption Rate")
ax2.yaxis.set_major_formatter(PercentFormatter(1.0))

plt.title("Adoption Rate & EV Registrations vs Total Superchargers")

# Combined legend
h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc="best")

plt.savefig(chart3_png, dpi=300, bbox_inches="tight")
plt.close()

print("✅ PNG charts created on Desktop.")


# -------- Chart 3 --------
# EV Registrations (left Y) & Adoption Rate (right Y)
# X-axis: TOP = Year (even 5-year ticks), BOTTOM = Superchargers (aligned)

plt.figure(figsize=(12, 7))

# True X axis = years (even spacing)
x_pos = np.arange(len(years))   # 0,1,2,... 26 positions

evs_series = evs
adopt_series = adopt
charger_series = chargers

# ======================================================
# -------- Chart 3 (Aligned Axes + Best-Fit Curves) ----
# ======================================================

plt.figure(figsize=(12, 7))

# True X axis = evenly spaced sequence mapped to years
x_pos = np.arange(len(years))   # 0,1,2,... 26 positions

evs_series = np.array(evs)
adopt_series = np.array(adopt)
charger_series = np.array(chargers)

# ======================================================
# 1. FIT BEST-FIT CURVES (CUBIC)
# ======================================================
coef_adopt = np.polyfit(charger_series, adopt_series, 3)
coef_evs = np.polyfit(charger_series, evs_series, 3)

poly_adopt = np.poly1d(coef_adopt)
poly_evs = np.poly1d(coef_evs)

# Smooth X for curves (but must map chargers → x_pos)
smooth_x_pos = np.linspace(min(x_pos), max(x_pos), 300)
smooth_chargers = np.interp(smooth_x_pos, x_pos, charger_series)

smooth_adopt = poly_adopt(smooth_chargers)
smooth_evs = poly_evs(smooth_chargers)

from sklearn.metrics import r2_score
print("\n=== BEST-FIT RESULTS ===")
print("Adoption vs Superchargers R²:", r2_score(adopt_series, poly_adopt(charger_series)))
print("EV Registrations vs Superchargers R²:", r2_score(evs_series, poly_evs(charger_series)))

# ======================================================
# 2. TOP AXIS (Years)
# ======================================================
ax_top = plt.gca()

# Scatter for EV Registrations
ax_top.scatter(
    x_pos, evs_series, 
    s=60, color="#1976D2",  # BLUE
    label="EV Registrations (scatter)"
)

# Best-fit EV curve
ax_top.plot(
    smooth_x_pos, smooth_evs,
    color="#0D47A1", linewidth=2.2, linestyle="--", 
    label="EV Registrations Best-Fit"
)

ax_top.set_ylabel("EV Registrations")
ax_top.grid(True)

# Even 5-year ticks
year_ticks = list(range(2024, 2051, 5))
tick_pos = [years.index(y) for y in year_ticks]

ax_top.set_xticks(tick_pos)
ax_top.set_xticklabels(year_ticks)
ax_top.set_xlabel("Year (Evenly Spaced 2024–2050)")

# ======================================================
# 3. RIGHT Y AXIS (Adoption %)
# ======================================================
ax_right = ax_top.twinx()

# Scatter for Adoption %
ax_right.scatter(
    x_pos, adopt_series, 
    s=60, marker="x", color="#D32F2F",  # RED
    label="Adoption Rate (scatter)"
)

# Best-fit Adoption curve
ax_right.plot(
    smooth_x_pos, smooth_adopt,
    color="#B71C1C", linewidth=2.2, linestyle="--",
    label="Adoption Rate Best-Fit"
)

ax_right.set_ylabel("Adoption Rate")
ax_right.yaxis.set_major_formatter(PercentFormatter(1.0))

# ======================================================
# 4. BOTTOM AXIS (Superchargers)
# ======================================================
ax_bottom = ax_top.twiny()
ax_bottom.set_xlim(ax_top.get_xlim())
ax_bottom.set_xticks(tick_pos)

bottom_labels = [charger_series[years.index(y)] for y in year_ticks]
ax_bottom.set_xticklabels(bottom_labels)
ax_bottom.set_xlabel("Total Superchargers (Aligned to Top-Year Scale)")

# ======================================================
# 5. TITLE + LEGEND
# ======================================================
plt.title("Adoption Rate & EV Registrations vs Total Superchargers (Aligned + Best-Fit Curves)")

# Combined legend
h1, l1 = ax_top.get_legend_handles_labels()
h2, l2 = ax_right.get_legend_handles_labels()
ax_top.legend(h1 + h2, l1 + l2, loc="upper left")

plt.savefig(chart3_png, dpi=300, bbox_inches="tight")
plt.close()

print("📈 Chart 3 with fits created:", chart3_png)

# ======================================================
#  INSERT NEW CHART 3 + FIT FORMULAS INTO EXCEL
# ======================================================

wb = load_workbook(excel_path)

# Create or replace Charts sheet
if "Charts" in wb.sheetnames:
    wb.remove(wb["Charts"])

ws = wb.create_sheet("Charts")

# ---------------- Insert Charts --------------------
img1 = XLImage(chart1_png)
img2 = XLImage(chart2_png)
img3 = XLImage(chart3_png)

ws.add_image(img1, "A1")
ws.add_image(img2, "A40")
ws.add_image(img3, "A80")

# ======================================================
# Write Polynomial Formulas + R² into Excel
# ======================================================
row_start = 120

ws[f"A{row_start}"] = "BEST-FIT CURVE FORMULAS"
ws[f"A{row_start}"].font = ws[f"A{row_start}"].font.copy(bold=True)

# -------- Adoption % Curve --------
row = row_start + 2
ws[f"A{row}"] = "1) Adoption Rate vs Superchargers (Cubic Polynomial)"
ws[f"A{row}"].font = ws[f"A{row}"].font.copy(bold=True)

# Equation: y = ax³ + bx² + cx + d
a, b, c, d = coef_adopt
row += 1
ws[f"A{row}"] = f"Equation:  y = {a:.10f}·x^3 + {b:.10f}·x^2 + {c:.10f}·x + {d:.10f}"

row += 1
ws[f"A{row}"] = f"R² = {r2_score(adopt_series, poly_adopt(charger_series)):.6f}"

# -------- EV Registrations Curve --------
row += 3
ws[f"A{row}"] = "2) EV Registrations vs Superchargers (Cubic Polynomial)"
ws[f"A{row}"].font = ws[f"A{row}"].font.copy(bold=True)

a2, b2, c2, d2 = coef_evs
row += 1
ws[f"A{row}"] = f"Equation:  y = {a2:.10f}·x^3 + {b2:.10f}·x^2 + {c2:.10f}·x + {d2:.10f}"

row += 1
ws[f"A{row}"] = f"R² = {r2_score(evs_series, poly_evs(charger_series)):.6f}"

# Save Excel file
wb.save(excel_path)

print("📊 Charts + formulas written into Excel successfully!")




Max chargers (should be ~1259): 1259
✅ PNG charts created on Desktop.

=== BEST-FIT RESULTS ===
Adoption vs Superchargers R²: 0.9647409885650519
EV Registrations vs Superchargers R²: 0.9647409885650519
📈 Chart 3 with fits created: /Users/judycheng/Desktop/chart3_adopt_evs_vs_chargers.png
📊 Charts + formulas written into Excel successfully!


/var/folders/6f/2tytnt59249dq7mgqy_lsckr0000gn/T/ipykernel_91833/2528777178.py:262: DeprecationWarning: Call to deprecated function copy (Use copy(obj) or cell.obj = cell.obj + other).
  ws[f"A{row_start}"].font = ws[f"A{row_start}"].font.copy(bold=True)
/var/folders/6f/2tytnt59249dq7mgqy_lsckr0000gn/T/ipykernel_91833/2528777178.py:267: DeprecationWarning: Call to deprecated function copy (Use copy(obj) or cell.obj = cell.obj + other).
  ws[f"A{row}"].font = ws[f"A{row}"].font.copy(bold=True)
/var/folders/6f/2tytnt59249dq7mgqy_lsckr0000gn/T/ipykernel_91833/2528777178.py:280: DeprecationWarning: Call to deprecated function copy (Use copy(obj) or cell.obj = cell.obj + other).
  ws[f"A{row}"].font = ws[f"A{row}"].font.copy(bold=True)


<Figure size 1200x700 with 0 Axes>

In [15]:
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score

# ======================================================
# FILE PATHS
# ======================================================
excel_path = "/Users/judycheng/Desktop/wa_county_ev_forecast_filled_mc.xlsx"
output_path = "/Users/judycheng/Desktop/wa_county_ev_forecast_fitted_by_county.xlsx"

df = pd.read_excel(excel_path)

years = list(range(2024, 2051))

# ======================================================
# Helper: cubic regression WITH intercept
# ======================================================
def cubic_with_intercept(x, y, label=""):
    x = np.array(x, float)
    y = np.array(y, float)

    # Fit cubic y = ax^3 + bx^2 + cx + d
    coef = np.polyfit(x, y, 3)
    y_pred = np.polyval(coef, x)
    r2 = r2_score(y, y_pred)

    a3, a2, a1, a0 = coef

    formula = (f"Cubic (R²={r2:.4f}): y = "
               f"{a3:.10f}*x^3 + {a2:.10f}*x^2 + "
               f"{a1:.10f}*x + {a0:.10f}")

    return formula, coef, r2


# ======================================================
# Build formulas for each county
# ======================================================
ev_formulas = []
adopt_formulas = []

for idx, row in df.iterrows():

    # ---- X = chargers from 2024–2050
    chargers = np.array([row.get(f"Superchargers_{y}", 0) for y in years], float)

    # ---- Y1 = EV Registrations
    evs = np.array([row.get(f"EVs_{y}", 0) for y in years], float)
    ev_formula, ev_coef, ev_r2 = cubic_with_intercept(chargers, evs, "EVs")

    # ---- Y2 = Adoption Rate
    adopt = np.array([row.get(f"Adoption_{y}", 0) for y in years], float)
    adopt_formula, adopt_coef, adopt_r2 = cubic_with_intercept(chargers, adopt, "Adoption")

    ev_formulas.append(ev_formula)
    adopt_formulas.append(adopt_formula)

# ======================================================
# Save back to Excel
# ======================================================
df["EVs_vs_SC_Formula"] = ev_formulas
df["Adopt_vs_SC_Formula"] = adopt_formulas

df.to_excel(output_path, index=False)

print("✅ Done! Using simple cubic-with-intercept models.")
print("📄 Saved:", output_path)


✅ Done! Using simple cubic-with-intercept models.
📄 Saved: /Users/judycheng/Desktop/wa_county_ev_forecast_fitted_by_county.xlsx


In [16]:
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit
from sklearn.metrics import r2_score

# ======================================================
# FILE PATHS
# ======================================================
input_path = "/Users/judycheng/Desktop/wa_county_ev_forecast_filled_mc.xlsx"
output_path = "/Users/judycheng/Desktop/wa_county_ev_forecast_fitted_by_county.xlsx"

df = pd.read_excel(input_path)
years = list(range(2024, 2051))


# ======================================================
# EXPONENTIAL MODEL FOR EV (ALWAYS POSITIVE)
# ======================================================
def exp_model(x, a, b):
    return a * np.exp(b * x)


def fit_exp(x, y):
    try:
        popt, _ = curve_fit(exp_model, x, y, p0=[y.max(), 0.001], maxfev=20000)
        y_pred = exp_model(x, *popt)
        return popt, r2_score(y, y_pred)
    except:
        return None, -999


# ======================================================
# EV BEST-FIT MODEL
# ======================================================
def best_ev_model(chargers, evs):
    x = np.array(chargers, float)
    y = np.array(evs, float)

    # Prevent negative EV or zeros (breaks exponential)
    y = np.where(y < 1, 1, y)

    models = {}

    # Linear
    try:
        coef = np.polyfit(x, y, 1)
        y_pred = np.polyval(coef, x)
        models["linear"] = (coef, r2_score(y, y_pred))
    except:
        models["linear"] = (None, -999)

    # Quadratic
    try:
        coef = np.polyfit(x, y, 2)
        y_pred = np.polyval(coef, x)
        models["quadratic"] = (coef, r2_score(y, y_pred))
    except:
        models["quadratic"] = (None, -999)

    # Cubic
    try:
        coef = np.polyfit(x, y, 3)
        y_pred = np.polyval(coef, x)
        models["cubic"] = (coef, r2_score(y, y_pred))
    except:
        models["cubic"] = (None, -999)

    # Exponential (ALWAYS positive)
    exp_coef, r2_exp = fit_exp(x, y)
    models["exponential"] = (exp_coef, r2_exp)

    # Pick best model
    best = max(models, key=lambda k: models[k][1])
    coef, r2_best = models[best]

    # ---- Return formula string ----
    if best == "linear":
        a1, a0 = coef
        return f"Linear (R²={r2_best:.4f}): y = {a1:.8f}*x + {a0:.8f}"

    if best == "quadratic":
        a2, a1, a0 = coef
        return (f"Quadratic (R²={r2_best:.4f}): y = "
                f"{a2:.10f}*x^2 + {a1:.10f}*x + {a0:.10f}")

    if best == "cubic":
        a3, a2, a1, a0 = coef
        return (f"Cubic (R²={r2_best:.4f}): y = "
                f"{a3:.10f}*x^3 + {a2:.10f}*x^2 + {a1:.10f}*x + {a0:.10f}")

    # Exponential
    a, b = coef
    return f"Exponential (R²={r2_best:.4f}): y = {a:.6f} * exp({b:.6f} * x)"


# ======================================================
# ADOPTION MODEL (NO CHANGE)
# ======================================================
def logistic(x, a, b):
    return 1 / (1 + np.exp(-(a * x + b)))

def best_adoption_model(x, y):
    x = np.array(x, float)
    y = np.array(y, float)

    if np.std(x) < 1e-6:
        return f"Constant: y = {float(np.mean(y)):.6f}"

    # Linear
    coef_lin = np.polyfit(x, y, 1)
    y_lin = np.polyval(coef_lin, x)
    r2_lin = r2_score(y, y_lin)

    # Quadratic
    coef_quad = np.polyfit(x, y, 2)
    y_quad = np.polyval(coef_quad, x)
    r2_quad = r2_score(y, y_quad)

    # Cubic
    coef_cubic = np.polyfit(x, y, 3)
    y_cubic = np.polyval(coef_cubic, x)
    r2_cubic = r2_score(y, y_cubic)

    # Logistic
    try:
        params, _ = curve_fit(logistic, x, y, p0=[0.005, -4], maxfev=20000)
        a, b = params
        y_log = logistic(x, a, b)
        r2_log = r2_score(y, y_log)
    except:
        r2_log = -999

    r2_dict = {
        "linear": r2_lin,
        "quadratic": r2_quad,
        "cubic": r2_cubic,
        "logistic": r2_log
    }
    best = max(r2_dict, key=r2_dict.get)

    if best == "logistic":
        return f"Logistic (R²={r2_log:.4f}): y = 1/(1+exp(-({a:.6f}*x + {b:.6f})))"

    if best == "cubic":
        a3,a2,a1,a0 = coef_cubic
        return f"Cubic (R²={r2_cubic:.4f}): y = {a3:.10f}*x^3 + {a2:.10f}*x^2 + {a1:.10f}*x + {a0:.10f}"

    if best == "quadratic":
        a2,a1,a0 = coef_quad
        return f"Quadratic (R²={r2_quad:.4f}): y = {a2:.10f}*x^2 + {a1:.10f}*x + {a0:.10f}"

    a1,a0 = coef_lin
    return f"Linear (R²={r2_lin:.4f}): y = {a1:.10f}*x + {a0:.10f}"


# ======================================================
# PROCESS EACH COUNTY
# ======================================================
EV_formulas = []
Adopt_formulas = []

for idx, row in df.iterrows():
    chargers = np.array([row.get(f"Superchargers_{y}", 0) for y in years])
    evs      = np.array([row.get(f"EVs_{y}",            0) for y in years])
    adopts   = np.array([row.get(f"Adoption_{y}",       0) for y in years])

    EV_formulas.append(best_ev_model(chargers, evs))
    Adopt_formulas.append(best_adoption_model(chargers, adopts))


df["EVs_vs_SC_Formula"] = EV_formulas
df["Adopt_vs_SC_Formula"] = Adopt_formulas

df.to_excel(output_path, index=False)

print("✅ Done! Independent county-level EV & Adoption formulas created.")


✅ Done! Independent county-level EV & Adoption formulas created.


In [17]:
import pandas as pd
import numpy as np
import re

# ------------------------------------------------------
# LOAD FILE WITH FORMULAS
# ------------------------------------------------------
path = "/Users/judycheng/Desktop/wa_county_ev_forecast_fitted_by_county.xlsx"
df = pd.read_excel(path)

# ------------------------------------------------------
# 1. Poly / cubic / quadratic / linear evaluator
# ------------------------------------------------------
def eval_poly(formula, x):
    # Extract terms like "-3.578990 * x^2"
    terms = re.findall(r"([\-0-9\.Ee]+)\s*\*\s*x\^?([0-9]*)", formula)

    if not terms:
        return None

    total = 0.0
    for coef, power in terms:
        coef = float(coef)
        power = int(power) if power != "" else 1
        total += coef * (x ** power)
    return total


# ------------------------------------------------------
# 2. Exponential evaluator (EV model)
#     y = a * exp(b*x)
# ------------------------------------------------------
def eval_exponential(formula, x):
    m = re.search(r"y\s*=\s*([0-9eE\.\-]+)\s*\*\s*exp\(\s*([0-9eE\.\-]+)\s*\*\s*x", formula)
    if not m:
        return None
    a = float(m.group(1))
    b = float(m.group(2))
    return a * np.exp(b * x)


# ------------------------------------------------------
# 3. Logistic evaluator (Adoption model)
#     y = 1/(1 + exp(-(a*x + b)))
# ------------------------------------------------------
def eval_logistic(formula, x):
    m = re.search(r"exp\(-\(([-0-9\.eE]+)\*x\s*\+\s*([-0-9\.eE]+)\)\)", formula)
    if not m:
        return None
    a = float(m.group(1))
    b = float(m.group(2))
    return 1 / (1 + np.exp(-(a*x + b)))


# ------------------------------------------------------
# 4. Constant evaluator
# ------------------------------------------------------
def eval_constant(formula):
    m = re.search(r"y\s*=\s*([-0-9\.eE]+)$", formula)
    return float(m.group(1)) if m else None


# ------------------------------------------------------
# 5. MASTER formula evaluator
# ------------------------------------------------------
def evaluate(formula, x):
    formula = str(formula)

    # logistic first
    val = eval_logistic(formula, x)
    if val is not None:
        return val

    # exponential
    val = eval_exponential(formula, x)
    if val is not None:
        return val

    # poly (linear, quad, cubic)
    val = eval_poly(formula, x)
    if val is not None:
        return val

    # constant
    val = eval_constant(formula)
    if val is not None:
        return val

    return np.nan  # fallback


# ------------------------------------------------------
# 6. Compute predictions using Superchargers_2050 column
# ------------------------------------------------------
EV_pred = []
Adopt_pred = []

for idx, row in df.iterrows():
    x = float(row.get("Superchargers_2050", 0))

    ev_formula = row["EVs_vs_SC_Formula"]
    adopt_formula = row["Adopt_vs_SC_Formula"]

    EV_pred.append(evaluate(ev_formula, x))
    Adopt_pred.append(evaluate(adopt_formula, x))

df["EV_Pred_2050"] = EV_pred
df["Adopt_Pred_2050"] = Adopt_pred

df.to_excel(path, index=False)

print("✅ Done! Predictions added:")
print("   → EV_Pred_2050")
print("   → Adopt_Pred_2050")
print("📄 Saved to:", path)


✅ Done! Predictions added:
   → EV_Pred_2050
   → Adopt_Pred_2050
📄 Saved to: /Users/judycheng/Desktop/wa_county_ev_forecast_fitted_by_county.xlsx


In [19]:
import pandas as pd
import numpy as np
import re

# ------------------------------------------------------
# LOAD FILE
# ------------------------------------------------------
path = "/Users/judycheng/Desktop/wa_county_ev_forecast_fitted_by_county.xlsx"
df = pd.read_excel(path)

# ------------------------------------------------------
# 1. Polynomial evaluator (cubic / quadratic / linear / constant)
# ------------------------------------------------------
def eval_poly(formula, x):
    """
    Evaluate polynomial after stripping everything before 'y ='.
    Supports:
      a*x^3 + b*x^2 + c*x + d
    """

    formula = str(formula)

    # keep only math: extract AFTER "y="
    m = re.search(r"y\s*=\s*(.*)", formula)
    if not m:
        return None

    expr = m.group(1)

    # extract x^n terms
    poly_terms = re.findall(r"([\-0-9\.Ee]+)\*x\^([0-9]+)", expr)

    # extract linear terms
    linear_terms = re.findall(r"([\-0-9\.Ee]+)\*x(?!\^)", expr)

    # extract constant term
    const_terms = re.findall(r"([\-0-9\.Ee]+)(?![\*x0-9Ee\.\-])", expr)

    total = 0.0

    for coef, power in poly_terms:
        total += float(coef) * (x ** int(power))

    for coef in linear_terms:
        total += float(coef) * x

    if const_terms:
        total += float(const_terms[-1])

    return total


# ------------------------------------------------------
# 2. Exponential evaluator
# ------------------------------------------------------
def eval_exponential(formula, x):
    m = re.search(
        r"y\s*=\s*([0-9eE\.\-]+)\s*\*\s*exp\(\s*([0-9eE\.\-]+)\s*\*\s*x",
        formula.replace(" ", "")
    )
    if not m:
        return None
    a = float(m.group(1))
    b = float(m.group(2))
    return a * np.exp(b * x)


# ------------------------------------------------------
# 3. Logistic evaluator
# ------------------------------------------------------
def eval_logistic(formula, x):
    f = formula.replace(" ", "")
    m = re.search(r"1/\(1\+exp\(-\(([-0-9\.eE]+)\*x\+([-0-9\.eE]+)\)\)\)", f)
    if not m:
        return None
    a = float(m.group(1))
    b = float(m.group(2))
    return 1 / (1 + np.exp(-(a*x + b)))


# ------------------------------------------------------
# 4. Constant evaluator
# ------------------------------------------------------
def eval_constant(formula):
    m = re.search(r"y\s*=\s*([-0-9\.Ee]+)$", formula)
    if m:
        return float(m.group(1))
    return None


# ------------------------------------------------------
# 5. MASTER evaluator (logistic → exponential → poly → constant)
# ------------------------------------------------------
def evaluate(formula, x):
    formula = str(formula)

    v = eval_logistic(formula, x)
    if v is not None:
        return v

    v = eval_exponential(formula, x)
    if v is not None:
        return v

    v = eval_poly(formula, x)
    if v is not None:
        return v

    v = eval_constant(formula)
    if v is not None:
        return v

    return np.nan


# ------------------------------------------------------
# 6. Compute predictions for each county
# ------------------------------------------------------
EV_pred = []
Adopt_pred = []

for idx, row in df.iterrows():
    x = float(row["Superchargers_2050"])
    EV_pred.append(evaluate(row["EVs_vs_SC_Formula"], x))
    Adopt_pred.append(evaluate(row["Adopt_vs_SC_Formula"], x))

df["EV_Pred_2050"] = EV_pred
df["Adopt_Pred_2050"] = Adopt_pred

df.to_excel(path, index=False)
print("✅ FINAL FIX APPLIED. Values now match manual calculation EXACTLY.")


✅ FINAL FIX APPLIED. Values now match manual calculation EXACTLY.


In [27]:
import pandas as pd
import numpy as np
import os
import re
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter, FuncFormatter

# ======================================================
# PATHS
# ======================================================
path = "/Users/judycheng/Desktop/wa_county_ev_forecast_fitted_by_county.xlsx"
df = pd.read_excel(path)

desktop = os.path.join(os.path.expanduser("~"), "Desktop")
output_folder = os.path.join(desktop, "county_charts_final")
os.makedirs(output_folder, exist_ok=True)

YEARS = list(range(2024, 2051))

# ======================================================
# EVALUATORS
# ======================================================
def eval_poly(formula, x):
    formula = str(formula)
    m = re.search(r"y\s*=\s*(.*)", formula)
    if not m:
        return None
    expr = m.group(1)

    poly_terms = re.findall(r"([\-0-9\.Ee]+)\*x\^([0-9]+)", expr)
    linear_terms = re.findall(r"([\-0-9\.Ee]+)\*x(?!\^)", expr)
    const_terms = re.findall(r"([\-0-9\.Ee]+)(?![\*x0-9Ee\.\-])", expr)

    total = 0.0
    for coef, power in poly_terms:
        total += float(coef) * (x ** int(power))
    for coef in linear_terms:
        total += float(coef) * x
    if const_terms:
        total += float(const_terms[-1])
    return total


def eval_exponential(formula, x):
    m = re.search(
        r"y\s*=\s*([0-9eE\.\-]+)\*exp\(([0-9eE\.\-]+)\*x",
        formula.replace(" ", "")
    )
    if not m:
        return None
    a = float(m.group(1))
    b = float(m.group(2))
    return a * np.exp(b * x)


def eval_logistic(formula, x):
    f = formula.replace(" ", "")
    m = re.search(r"1/\(1\+exp\(-\(([-0-9\.Ee]+)\*x\+([-0-9\.Ee]+)\)\)\)", f)
    if not m:
        return None
    a = float(m.group(1))
    b = float(m.group(2))
    return 1 / (1 + np.exp(-(a*x + b)))


def eval_constant(formula):
    m = re.search(r"y\s*=\s*([-0-9\.Ee]+)$", str(formula))
    if m:
        return float(m.group(1))
    return None


def evaluate(formula, x):
    formula = str(formula)

    v = eval_logistic(formula, x)
    if v is not None:
        return v

    v = eval_exponential(formula, x)
    if v is not None:
        return v

    v = eval_poly(formula, x)
    if v is not None:
        return v

    v = eval_constant(formula)
    if v is not None:
        return v

    return np.nan


# ======================================================
# FORMATTERS
# ======================================================
def millions_formatter(x, pos):
    if abs(x) >= 1_000_000:
        return f"{x/1_000_000:.1f}M"
    return f"{int(x):,}"


# ======================================================
# CHART GENERATOR
# ======================================================
def generate_chart(row):

    county = row["County"]

    # Build curve domain
    chargers = np.array([row[f"Superchargers_{y}"] for y in YEARS], float)
    xmin, xmax = chargers.min(), chargers.max()

    if xmin == xmax:
        xmin = xmin * 0.9
        xmax = xmax * 1.1
        if xmin == xmax:
            xmin -= 1
            xmax += 1

    x_line = np.linspace(xmin, xmax, 400)

    # Evaluate curves
    y_evs = [evaluate(row["EVs_vs_SC_Formula"], x) for x in x_line]
    y_adopt = [evaluate(row["Adopt_vs_SC_Formula"], x) for x in x_line]

    # 2024 baseline
    sc24 = row["Superchargers_2024"]
    ev24 = row["EVs_2024"]
    ad24 = row["Adoption_2024"]

    plt.figure(figsize=(10, 6))
    ax1 = plt.gca()
    ax2 = ax1.twinx()

    # Curves (improved styling)
    ax1.plot(x_line, y_evs, color="blue", linewidth=4, alpha=0.95, label="EV Registrations")
    ax2.plot(x_line, y_adopt, color="red", linewidth=2, alpha=0.55, label="Adoption Rate")

    # 2024 dots (improved styling)
    ax1.scatter(sc24, ev24, color="blue", s=160, edgecolor="black", linewidth=1.2, zorder=5)
    ax2.scatter(sc24, ad24, color="red", s=110, edgecolor="black", alpha=0.55, linewidth=1.0, zorder=5)


    # Labels
    ax1.set_xlabel("Total Superchargers")
    ax1.set_ylabel("EV Registrations", color="blue")
    ax2.set_ylabel("Adoption Rate", color="red")
    ax1.yaxis.set_major_formatter(FuncFormatter(millions_formatter))
    ax2.yaxis.set_major_formatter(PercentFormatter(1.0))

    # ==============================
    # CLEAN TITLES (fixed)
    # ==============================
    if county == "TOTAL":
        title = "Washington: EV Registrations & Adoption vs Superchargers"
    else:
        title = f"{county}: EV Registrations & Adoption vs Superchargers"

    plt.title(title)
    ax1.grid(True, linestyle="--", alpha=0.6)

    # ==============================
    # LEGEND WITH EXPLANATION
    # ==============================

    curve_ev = plt.Line2D([], [], color='blue', linewidth=4, label='EV Registrations (Curve)')
    curve_ad = plt.Line2D([], [], color='red', linewidth=2, alpha=0.55, label='Adoption Rate (Curve)')

    dot_ev = plt.Line2D([], [], color='blue', marker='o', markersize=12, linewidth=0,
                        markeredgecolor='black', label='2024 EV Registration (Actual)')
    dot_ad = plt.Line2D([], [], color='red', marker='o', markersize=10, linewidth=0,
                        alpha=0.55, markeredgecolor='black', label='2024 Adoption Rate (Actual)')

    ax1.legend(
        handles=[curve_ev, curve_ad, dot_ev, dot_ad],
        loc='upper left',
        frameon=True,
        facecolor='white',
        edgecolor='black',
        fontsize=10
    )


    # Save image exactly as CountyName.png
    out_path = os.path.join(output_folder, f"{county}.png")
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()

    print(f"✅ Saved: {county}.png")


# ======================================================
# RUN FOR ALL COUNTIES
# ======================================================
for idx, row in df.iterrows():
    generate_chart(row)

print("\n🎉 All charts created successfully!")
print("📁 Saved to:", output_folder)


✅ Saved: Adams.png
✅ Saved: Asotin.png
✅ Saved: Benton.png
✅ Saved: Chelan.png
✅ Saved: Clallam.png
✅ Saved: Clark.png
✅ Saved: Columbia.png
✅ Saved: Cowlitz.png
✅ Saved: Douglas.png
✅ Saved: Ferry.png
✅ Saved: Franklin.png
✅ Saved: Garfield.png
✅ Saved: Grant.png
✅ Saved: Grays Harbor.png
✅ Saved: Island.png
✅ Saved: Jefferson.png
✅ Saved: King.png
✅ Saved: Kitsap.png
✅ Saved: Kittitas.png
✅ Saved: Klickitat.png
✅ Saved: Lewis.png
✅ Saved: Lincoln.png
✅ Saved: Mason.png
✅ Saved: Okanogan.png
✅ Saved: Pacific.png
✅ Saved: Pend Oreille.png
✅ Saved: Pierce.png
✅ Saved: San Juan.png
✅ Saved: Skagit.png
✅ Saved: Skamania.png
✅ Saved: Snohomish.png
✅ Saved: Spokane.png
✅ Saved: Stevens.png
✅ Saved: Thurston.png
✅ Saved: Wahkiakum.png
✅ Saved: Walla Walla.png
✅ Saved: Whatcom.png
✅ Saved: Whitman.png
✅ Saved: Yakima.png
✅ Saved: TOTAL.png

🎉 All charts created successfully!
📁 Saved to: /Users/judycheng/Desktop/county_charts_final
